# Autocorrelation-bias fix — OLD vs Stage-1 vs Stage-2 MultiModalSCVI

Benchmarks fixes for the spatial **autocorrelation bias** (spatial latent collapsing onto the
abundance latent) against the established baseline, on `adata_all_annotated`.

- **OLD** (`old_top500`) — current baseline: top-500 high-variance spatial pairs, per-cell decoder
  variance, no free-bits.
- **Stage 1 / surgical** (`new_white`) — whitened/decorrelated spatial PCs
  (`build_spatial_pca_obsm`, zscore + whiten), fixed unit-variance Normal decoder on the spatial
  modality (`decoder_dispersions=[…, 'fixed']`), per-dim KL free-bits (`free_bits=0.5`), and the
  POE-weighting fix.
- **Stage 2 / structural** (`new_private`) — Stage-1 **plus** a dedicated spatial-private latent
  subspace (`n_private=[0, 10]`): dims encoded only from spatial and decoded only to spatial, which
  the abundance-dominated shared latent cannot override.

Compares: modality weights, spatial/abundance **Moran's I** per latent, reconstruction PPC, and scIB
metrics. Train on a GPU kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
PIXELGEN_ROOT = '/home/projects/nyosef/zvise/PixelGen/PixelGen'
if PIXELGEN_ROOT not in sys.path:
    sys.path.append(PIXELGEN_ROOT)
if '/home/projects/nyosef/zvise/PixelGen/' not in sys.path:
    sys.path.append('/home/projects/nyosef/zvise/PixelGen/')

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import torch
import scvi
from matplotlib import pyplot as plt

from multimodalvi import MultiModalSCVI
from enums import AggMethod, D
from metrics import distr_autocorrelation_in_latent
from utils import build_spatial_pca_obsm, plot_composite_ppc, get_dense, calculate_metrics

scvi.settings.seed = 0
torch.set_float32_matmul_precision('high')
sc.set_figure_params(figsize=(5, 3), frameon=False)
print('scvi', scvi.__version__, '| cuda', torch.cuda.is_available())

## 1. Load data

`adata_all_annotated.h5ad`: 15782 cells × 159 markers, abundance layers (`arcsinh`/`clr`/`log1p`),
full `obsm['spatial_asinh5']` (~12k pairs) + `obsm['spatial_asinh5_top500var']`, annotations
`cell_type_annot` and batch `cell_system`.

In [ ]:
ABUNDANCE_LAYER = 'arcsinh'
BATCH_KEY       = 'cell_system'
BIO_KEY         = 'cell_type_annot'
N_LATENT        = 20

ADATA_PATH = f'{PIXELGEN_ROOT}/New_Data/cache/adata_all_annotated.h5ad'
adata = sc.read_h5ad(ADATA_PATH)
print(adata)
print('\nobsm:', list(adata.obsm.keys()))
print('\n' + BIO_KEY + ':');   print(adata.obs[BIO_KEY].value_counts())
print('\n' + BATCH_KEY + ':'); print(adata.obs[BATCH_KEY].value_counts())

## 2a. Build the whitened spatial modality

In [ ]:
# NEW representation: whitened, decorrelated spatial PCs (generalizes the top-500 trick).
# zscore + whiten => ~unit-variance, decorrelated features, so a fixed unit-variance Normal
# decoder gives inverse-variance-correct NLL and can't "cheat" by predicting the population mean.
SPATIAL_SOURCE = 'spatial_asinh5'
TOP500_KEY     = 'spatial_asinh5_top500var'   # OLD baseline rep (already in obsm)

KEY_WHITE = build_spatial_pca_obsm(
    adata, source_key=SPATIAL_SOURCE, n_components=64,
    standardize='zscore', whiten=True, target_key='spatial_asinh5_pca64_white',
)

assert isinstance(adata.obsm[KEY_WHITE], pd.DataFrame)
assert TOP500_KEY in adata.obsm
print('OLD spatial:', adata.obsm[TOP500_KEY].shape, '| NEW spatial:', adata.obsm[KEY_WHITE].shape)

## 2b. Training helper

In [ ]:
TRAIN_KWARGS = dict(max_epochs=200, batch_size=512, early_stopping=True)

def train_variant(setup_kwargs, model_kwargs, name):
    """Train one MultiModalSCVI on its own adata copy (each run owns its scvi registry)."""
    a = adata.copy()
    MultiModalSCVI.setup_anndata(a, **setup_kwargs)
    model = MultiModalSCVI(a, **model_kwargs)
    print(f'>>> training {name}: input_ds={model.input_ds}')
    model.train(**TRAIN_KWARGS)
    return a, model

## 3. Train models — OLD baseline vs Stage-1 (surgical) vs Stage-2 (private latent)

Four runs, identical architecture/optimizer; only the spatial representation + the fix knobs vary:

| run | spatial input | decoder variance | free bits | private dims |
|-----|---------------|------------------|-----------|--------------|
| `abundance_only` | — | per-cell | 0 | — |
| `old_top500` | top-500 high-var pairs | per-cell (`gene-cell`) | 0 | 0 |
| `new_white` | 64 whitened PCs | fixed unit (`fixed`) on spatial | 0.5 | 0 |
| `new_private` | 64 whitened PCs | fixed unit on spatial | 0.5 | **10 (spatial-only)** |

The POE-weighting fix applies to all multimodal runs (it was previously a no-op). `new_private` adds a
spatial-private latent subspace on top of the Stage-1 fixes.

In [ ]:
DECODER_KWARGS = dict(decoder_param_eps=1e-2, decoder_activation='exp')

# --- abundance-only baseline (single modality) ---
abn_setup = dict(layer=ABUNDANCE_LAYER, extra_modality_keys=[], n_modalities=1, batch_key=BATCH_KEY)
abn_model_kwargs = dict(
    n_latent=N_LATENT, n_hidden=128, n_layers=2, dropout_rate=0.1,
    distrs=[D.Normal], unimodal_kl=True, joint_kl=False, decoder_kwargs=DECODER_KWARGS,
)

# --- OLD model: established baseline (top-500 spatial, per-cell variance, no free-bits) ---
old_setup = dict(layer=ABUNDANCE_LAYER, extra_modality_keys=[TOP500_KEY], n_modalities=2, batch_key=BATCH_KEY)
old_model_kwargs = dict(
    n_latent=N_LATENT, n_hidden=128, n_layers=2, dropout_rate=0.1,
    distrs=[D.Normal, D.Normal], experts_method='POE', loss_weights='auto',
    unimodal_kl=True, joint_kl=False, batch_mask=[False, True], decoder_kwargs=DECODER_KWARGS,
)

# --- NEW model (Stage 1, surgical): whitened spatial, fixed unit variance, KL free-bits ---
new_setup = dict(layer=ABUNDANCE_LAYER, extra_modality_keys=[KEY_WHITE], n_modalities=2, batch_key=BATCH_KEY)
new_model_kwargs = dict(
    n_latent=N_LATENT, n_hidden=128, n_layers=2, dropout_rate=0.1,
    distrs=[D.Normal, D.Normal], experts_method='POE', loss_weights='auto',
    unimodal_kl=True, joint_kl=False, batch_mask=[False, True],
    decoder_dispersions=['gene-cell', 'fixed'],  # fixed unit variance on whitened spatial
    free_bits=0.5,                               # per-dim KL floor -> stops spatial collapse
    decoder_kwargs=DECODER_KWARGS,
)

# --- PRIVATE model (Stage 2, structural): Stage-1 + a dedicated spatial-private latent ---
# n_private=[0, N_PRIVATE] adds N_PRIVATE dims encoded ONLY from spatial and decoded ONLY to
# spatial, so the abundance-dominated POE-shared latent cannot override the spatial signal.
N_PRIVATE = 10
private_setup = dict(layer=ABUNDANCE_LAYER, extra_modality_keys=[KEY_WHITE], n_modalities=2, batch_key=BATCH_KEY)
private_model_kwargs = dict(
    n_latent=N_LATENT, n_hidden=128, n_layers=2, dropout_rate=0.1,
    distrs=[D.Normal, D.Normal], experts_method='POE', loss_weights='auto',
    unimodal_kl=True, joint_kl=False, batch_mask=[False, True],
    decoder_dispersions=['gene-cell', 'fixed'],
    free_bits=0.5,
    n_private=[0, N_PRIVATE],
    decoder_kwargs=DECODER_KWARGS,
)

runs = {}
runs['abundance_only'] = train_variant(abn_setup, abn_model_kwargs, 'abundance_only')
runs['old_top500']     = train_variant(old_setup, old_model_kwargs, 'old_top500')
runs['new_white']      = train_variant(new_setup, new_model_kwargs, 'new_white')
runs['new_private']    = train_variant(private_setup, private_model_kwargs, 'new_private')

## 4. Extract latents

In [ ]:
a_abn, m_abn = runs['abundance_only']
a_old, m_old = runs['old_top500']
a_new, m_new = runs['new_white']
a_prv, m_prv = runs['new_private']

# One shared view holding every latent (all are copies of the same cells, identical obs order).
adata_ac = adata.copy()
adata_ac.obsm['z_joint__abundance_only'] = m_abn.get_latent_representation(a_abn, modality='joint')
adata_ac.obsm['z_joint__old_top500']     = m_old.get_latent_representation(a_old, modality='joint')
adata_ac.obsm['z_joint__new_white']      = m_new.get_latent_representation(a_new, modality='joint')
adata_ac.obsm['z_joint__new_private']    = m_prv.get_latent_representation(a_prv, modality='joint')
adata_ac.obsm['z_spatial__old_top500']   = m_old.get_latent_representation(a_old, modality=TOP500_KEY)
adata_ac.obsm['z_spatial__new_white']    = m_new.get_latent_representation(a_new, modality=KEY_WHITE)
# Spatial head of the private model = spatial encoder output (shared dims + the N_PRIVATE private dims).
adata_ac.obsm['z_spatial__new_private']  = m_prv.get_latent_representation(a_prv, modality=KEY_WHITE)

latent_keys = ['z_joint__abundance_only', 'z_joint__old_top500', 'z_joint__new_white', 'z_joint__new_private',
               'z_spatial__old_top500', 'z_spatial__new_white', 'z_spatial__new_private']
latent_names = ['abundance_only', 'old_top500_joint', 'new_white_joint', 'new_private_joint',
                'old_top500_spatialhead', 'new_white_spatialhead', 'new_private_spatialhead']

print('new_private spatial-head dim:', adata_ac.obsm['z_spatial__new_private'].shape,
      '(= N_LATENT shared + N_PRIVATE)')

# Modality weights: with the POE fix these are no longer a no-op. A non-trivial spatial weight
# confirms the spatial expert actually contributes to the joint posterior.
print('old_top500 weights:', m_old.get_weights())
print('new_white  weights:', m_new.get_weights())
print('new_private weights:', m_prv.get_weights())

## 5. Autocorrelation-bias benchmark

The bias symptom: spatial-feature autocorrelation in the latent neighbor graph looks the same
regardless of whether spatial input was used (the joint latent collapses to the abundance latent).
Mirrors `PBMSC/benchmark_autocorrelation.ipynb` / `spatial_pca_benchmark.ipynb`: build a kNN graph
per latent and compute Moran's I per feature. **Success** = the new model's joint *and* spatial-head
latents give higher spatial Moran's I than the old model and the `abundance_only` baseline, and the
spatial head is no longer interchangeable with the abundance latent.

In [ ]:
# Moran's I of spatial features (common ground truth = top500var) under each latent's kNN graph,
# and of abundance features. Higher spatial Moran's I for the new model => the latent organizes
# cells along genuinely spatial axes instead of echoing the abundance structure.
autocorr_spatial = distr_autocorrelation_in_latent(
    adata_ac, latent_keys=latent_keys, names=latent_names,
    rep_key=TOP500_KEY, pca_kwargs={'n_comps': 15},
)
autocorr_abundance = distr_autocorrelation_in_latent(
    adata_ac, latent_keys=latent_keys, names=latent_names,
    rep_key=ABUNDANCE_LAYER, pca_kwargs={'n_comps': 15},
)
print('spatial:', autocorr_spatial.shape, '| abundance:', autocorr_abundance.shape)

In [ ]:
import seaborn as sns

# Summary: mean/median Moran's I across features, per latent and feature type.
summary_rows = []
for label, df in [('Spatial', autocorr_spatial), ('Abundance', autocorr_abundance)]:
    for nm in latent_names:
        s = df[df['latent'] == nm]['morans']
        summary_rows.append({'Feature type': label, 'Model': nm,
                             'Mean': s.mean(), 'Median': s.median(), 'Std': s.std()})
summary_df = pd.DataFrame(summary_rows)
display(summary_df.pivot(index='Model', columns='Feature type', values=['Mean', 'Median']))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(data=autocorr_spatial, x='morans', hue='latent', kde=True, stat='density',
             common_norm=False, alpha=0.4, ax=axes[0])
axes[0].set_title(f"Spatial Moran's I (rep={TOP500_KEY})"); axes[0].set_xlabel("Moran's I")
sns.histplot(data=autocorr_abundance, x='morans', hue='latent', kde=True, stat='density',
             common_norm=False, alpha=0.4, ax=axes[1])
axes[1].set_title(f"Abundance Moran's I (rep={ABUNDANCE_LAYER})"); axes[1].set_xlabel("Moran's I")
plt.tight_layout(); plt.show()

## 6. Reconstruction quality (posterior predictive checks)

`get_normalized_expression` reconstructs each model's own inputs. Abundance (`arcsinh`, 159 markers)
is the same target for both models — directly comparable. The spatial target differs (old = 500
top-variance pairs, new = 64 whitened PCs), so spatial PPC measures each model's **self**-reconstruction
quality, not a head-to-head on identical features. The "feature dependency" symptom shows up as the
old model reconstructing well only on the highest-variance pairs.

In [ ]:
all_metrics = []
ppc_specs = [
    ('old_top500',  runs['old_top500'],  TOP500_KEY),
    ('new_white',   runs['new_white'],   KEY_WHITE),
    ('new_private', runs['new_private'], KEY_WHITE),
]
for name, (a, model), sp_key in ppc_specs:
    out = model.get_normalized_expression(
        adata=a,
        return_mean_expression=True,
        return_l2_error=False,
        return_px_distrs=False,
        return_numpy=True,
    )
    ab_obs = get_dense(a.layers[ABUNDANCE_LAYER]); ab_gen = get_dense(out['exprs'][ABUNDANCE_LAYER])
    all_metrics += calculate_metrics(ab_obs, ab_gen, name, 'Abundance')
    sp_obs = get_dense(a.obsm[sp_key]);            sp_gen = get_dense(out['exprs'][sp_key])
    all_metrics += calculate_metrics(sp_obs, sp_gen, name, 'Spatial')

metrics_df = pd.DataFrame(all_metrics)
model_names = ['old_top500', 'new_white', 'new_private']
plot_composite_ppc('Abundance', metrics_df, scatter_color='cornflowerblue', model_names=model_names); plt.show()
plot_composite_ppc('Spatial',   metrics_df, scatter_color='lightgreen',     model_names=model_names); plt.show()

## 7. scIB metrics (bio conservation / batch correction)

`bio = cell_type_annot`, `batch = cell_system`. Includes both joint latents and both spatial-head
latents. The fix should keep (or improve) bio conservation while the spatial head carries *distinct*
structure rather than echoing the abundance latent.

In [ ]:
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

embedding_keys = [
    'z_joint__abundance_only',
    'z_joint__old_top500',
    'z_joint__new_white',
    'z_joint__new_private',
    'z_spatial__old_top500',
    'z_spatial__new_white',
    'z_spatial__new_private',
]
bm = Benchmarker(
    adata_ac,
    batch_key=BATCH_KEY,
    label_key=BIO_KEY,
    embedding_obsm_keys=embedding_keys,
    bio_conservation_metrics=BioConservation(),
    batch_correction_metrics=BatchCorrection(),
)
bm.benchmark()
bm.plot_results_table(min_max_scale=False)